# Notebook 10 -- Shor's Algorithm and Quantum Counting

Shor's algorithm is the most celebrated result in quantum computing: it factors
integers in polynomial time, breaking the RSA cryptosystem that underpins most
of today's internet security. Where the best known classical factoring algorithm
(the general number field sieve) runs in sub-exponential time
$O(\exp(n^{1/3} (\log n)^{2/3}))$, Shor's algorithm runs in $O(n^3)$ using
quantum phase estimation.

By the end of this notebook you will be able to:

1. **Describe** how Shor's algorithm reduces factoring to period finding.
2. **Implement** integer factoring using Goqu's shor package.
3. **Explain** quantum counting and its relationship to Grover's algorithm.
4. **Compare** standard and iterative amplitude estimation.

In this notebook we will:

1. Understand how Shor's algorithm reduces factoring to period finding.
2. Factor small integers using Goqu's `shor` package.
3. Explore **quantum counting** -- estimating the number of solutions to a
   search problem using QPE on the Grover iterate.
4. Explore **amplitude estimation** -- a generalization that estimates the
   probability amplitude of a "good" subspace.
5. Compare standard and iterative amplitude estimation.

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/ampest"
	"github.com/splch/goqu/algorithm/counting"
	"github.com/splch/goqu/algorithm/grover"
	"github.com/splch/goqu/algorithm/shor"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Factoring Problem

Given a composite integer $N$, find two non-trivial factors $p$ and $q$ such
that $N = p \times q$. This is believed to be classically hard -- no
polynomial-time algorithm is known, and RSA encryption relies on this hardness.

For example, given $N = 15$, the factors are $3$ and $5$. For small numbers
this is trivial, but for numbers with hundreds of digits, classical algorithms
become infeasible.

## How Shor's Algorithm Works

Shor's algorithm converts factoring into **period finding** through the
following steps:

1. **Choose a random base** $a$ coprime to $N$.
2. **Find the period** $r$ of $f(x) = a^x \bmod N$ -- the smallest $r$ such
   that $a^r \equiv 1 \pmod{N}$.
3. **Extract factors**: if $r$ is even, compute
   $\gcd(a^{r/2} \pm 1, N)$ which, with high probability, yields a
   non-trivial factor.

The quantum speedup lives in step 2. Classically, finding the period requires
exponentially many evaluations of $f$. Quantum mechanically, we use **Quantum
Phase Estimation (QPE)** on the modular exponentiation unitary
$U|y\rangle = |ay \bmod N\rangle$.

The eigenvalues of $U$ are $e^{2\pi i s/r}$ for $s = 0, \ldots, r-1$, so QPE
measures a phase $\varphi \approx s/r$. A **continued fraction expansion** of
$\varphi$ recovers the period $r$, from which we compute the factors.

### Circuit Structure

The quantum circuit consists of:

1. A **phase register** of $t$ qubits initialized in superposition.
2. A **target register** of $\lceil\log_2 N\rceil$ qubits initialized to $|1\rangle$.
3. **Controlled modular exponentiation**: $\text{C-}U^{2^k}$ for each phase qubit $k$.
4. **Inverse QFT** on the phase register to extract the phase.
5. **Measurement** of the phase register, followed by classical post-processing.

In [ ]:
%%
ctx := context.Background()

// Factor N = 15 using Shor's algorithm
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("N = 15 = %d x %d\n", result.Factors[0], result.Factors[1])
fmt.Printf("Period: %d, Base: %d, Attempts: %d\n", result.Period, result.Base, result.Attempts)
fmt.Printf("\nVerification: %d x %d = %d\n",
	result.Factors[0], result.Factors[1],
	result.Factors[0]*result.Factors[1])

In [ ]:
%%
ctx := context.Background()

// Run Shor's on N=15 again and display the order-finding circuit
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

if result.Circuit != nil {
	fmt.Println("Order-finding circuit (QPE + modular exponentiation):")
	fmt.Println(draw.String(result.Circuit))
} else {
	fmt.Println("Factor found classically (no circuit needed for this attempt).")
}

In [ ]:
%%
ctx := context.Background()

// Run Shor's on N=15 and display the QPE measurement histogram.
// The circuit measures only the phase register; peaks correspond to
// phases s/r that encode the period r.
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

if result.Circuit != nil {
	// Re-run the circuit to get measurement counts for visualization
	nTarget := int(math.Ceil(math.Log2(float64(15))))
	nPhase := 2 * nTarget
	nTotal := nPhase + nTarget
	sim := statevector.New(nTotal)
	counts, err := sim.Run(result.Circuit, 1000)
	if err != nil {
		panic(err)
	}
	fmt.Println("QPE measurement histogram for N=15:")
	fmt.Println("Peaks at multiples of 1/r reveal the period r.")
	svg := viz.Histogram(counts, viz.WithTitle("Shor's Algorithm: QPE Output (N=15)"))
	gonbui.DisplayHTML(svg)
} else {
	fmt.Println("Factor found classically -- no quantum circuit to visualize.")
}

## Quantum Counting

**Quantum counting** answers the question: *how many solutions does a search
problem have?* This is useful when we need to know the number of marked states
$M$ in a search space of size $N = 2^n$ before running Grover's search (since
the optimal iteration count depends on $M$).

The algorithm works by applying **QPE to the Grover iterate** $Q$. The Grover
operator has eigenvalues $e^{\pm 2i\theta}$ where $\sin^2(\theta) = M/N$.
QPE estimates $\theta$, from which we recover:

$$M = N \sin^2(\theta)$$

With $t$ phase bits, the estimate achieves $t$ bits of precision. This gives a
quadratic speedup over classical counting, which requires $O(N)$ oracle queries.

In [ ]:
%%
ctx := context.Background()

// Use quantum counting to estimate the number of solutions
// to a Grover oracle. We mark states |01> and |10> (2 solutions
// out of 4 total states in a 2-qubit search space).
oracle := grover.PhaseOracle([]int{0b01, 0b10}, 2)

result, err := counting.Run(ctx, counting.Config{
	NumQubits:    2,
	Oracle:       oracle,
	NumPhaseBits: 4,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Estimated count: %.1f (expected: 2)\n", result.Count)
fmt.Printf("Phase: %.4f\n", result.Phase)
fmt.Println("Measurement counts:", result.Counts)

svg := viz.Histogram(result.Counts, viz.WithTitle("Quantum Counting: 2 Solutions in 4 States"))
gonbui.DisplayHTML(svg)

## Amplitude Estimation

**Amplitude estimation** generalizes quantum counting. Instead of counting
the number of solutions, it estimates the **probability amplitude** $a$ of a
"good" subspace prepared by a state-preparation circuit $A$.

Given:
- A state-preparation circuit $A$ that creates $A|0\rangle = \sqrt{1-a^2}|\psi_0\rangle + a|\psi_1\rangle$
- An oracle $S_f$ that marks the "good" states $|\psi_1\rangle$

Standard amplitude estimation builds the **Grover iterate**
$Q = A \cdot S_0 \cdot A^\dagger \cdot S_f$ and applies QPE to estimate the
phase $\theta$ where $a = \sin(\pi\theta)$.

This is a core subroutine in:
- **Monte Carlo methods** (quadratic speedup for financial derivative pricing)
- **Quantum machine learning** (estimating inner products)
- **Optimization** (evaluating objective functions)

In [ ]:
%%
ctx := context.Background()

// Amplitude estimation: estimate the amplitude of |11> in the
// uniform superposition H|0>H|0> = (|00>+|01>+|10>+|11>)/2.
// The amplitude of |11> is 1/2, so probability = 1/4.
statePrep, err := builder.New("prep", 2).H(0).H(1).Build()
if err != nil {
	panic(err)
}

oracle := grover.PhaseOracle([]int{0b11}, 2)

result, err := ampest.Run(ctx, ampest.Config{
	StatePrep:    statePrep,
	Oracle:       oracle,
	NumQubits:    2,
	NumPhaseBits: 4,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Amplitude: %.4f (expected: 0.5000)\n", result.Amplitude)
fmt.Printf("Probability: %.4f (expected: 0.2500)\n", result.Probability)
fmt.Printf("Phase: %.4f\n", result.Phase)
fmt.Println("Measurement counts:", result.Counts)

svg := viz.Histogram(result.Counts, viz.WithTitle("Amplitude Estimation: P(|11>) in Uniform State"))
gonbui.DisplayHTML(svg)

## Predict, Then Verify

**Question:** What are the factors of $N = 21$?

**Prediction:** $21 = 3 \times 7$. Shor's algorithm should find these factors
by discovering the period of $a^x \bmod 21$ for some random base $a$.

For example, with $a = 2$:
- $2^1 \equiv 2$, $2^2 \equiv 4$, $2^3 \equiv 8$, $2^4 \equiv 16$,
  $2^5 \equiv 11$, $2^6 \equiv 1 \pmod{21}$
- Period $r = 6$, which is even.
- $\gcd(2^3 - 1, 21) = \gcd(7, 21) = 7$ and
  $\gcd(2^3 + 1, 21) = \gcd(9, 21) = 3$.

Let's verify with Goqu.

In [ ]:
%%
ctx := context.Background()

// Factor N = 21
result, err := shor.Run(ctx, shor.Config{
	N:     21,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("N = 21 = %d x %d\n", result.Factors[0], result.Factors[1])
fmt.Printf("Period: %d, Base: %d, Attempts: %d\n", result.Period, result.Base, result.Attempts)
fmt.Printf("\nVerification: %d x %d = %d\n",
	result.Factors[0], result.Factors[1],
	result.Factors[0]*result.Factors[1])
fmt.Println("\nPrediction confirmed: 21 = 3 x 7.")

---

## Exercises

### Exercise 1 -- Factor N = 35

Run Shor's algorithm on $N = 35$. Verify that the returned factors are correct
and print the period and base that the algorithm discovered.

In [ ]:
%%
// Exercise 1: Factor N = 35
// Expected factors: 5 and 7
//
// TODO: Run Shor's algorithm on N = 35
// result, err := shor.Run(ctx, shor.Config{
//     N:     ???,
//     Shots: 1000,
// })
//
// TODO: Print the factors, period, base, and number of attempts
// fmt.Printf("N = 35 = %d x %d\n", result.Factors[0], result.Factors[1])
//
// TODO: Verify correctness by checking result.Factors[0]*result.Factors[1] == 35
ctx := context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

### Exercise 2 -- Compare Standard vs Iterative Amplitude Estimation

Standard amplitude estimation uses QPE with a full ancilla register, building
an $O(2^t)$-depth circuit. **Iterative amplitude estimation** avoids the large
ancilla register by using a single ancilla qubit and progressively doubling the
Grover power, achieving similar precision with lower circuit depth.

Run both methods on the same problem (estimating the amplitude of $|11\rangle$
in the uniform 2-qubit superposition) and compare their accuracy.

In [ ]:
%%
// Exercise 2: Standard vs Iterative Amplitude Estimation
// Estimate the amplitude of |11> in the uniform 2-qubit superposition.
// The amplitude of |11> is 1/2, so probability = 1/4.
//
// TODO: Build the state preparation circuit (H on both qubits)
// statePrep, err := builder.New("prep", 2).H(0).H(1).Build()
//
// TODO: Create a phase oracle marking |11>
// oracle := grover.PhaseOracle([]int{???}, 2)
//
// TODO: Run standard amplitude estimation with 4 phase bits
// stdResult, err := ampest.Run(ctx, ampest.Config{
//     StatePrep:    statePrep,
//     Oracle:       oracle,
//     NumQubits:    ???,
//     NumPhaseBits: ???,
//     Shots:        1000,
// })
//
// TODO: Run iterative amplitude estimation with 4 max iterations
// iterResult, err := ampest.RunIterative(ctx, ampest.IterativeConfig{
//     StatePrep: statePrep,
//     Oracle:    oracle,
//     NumQubits: ???,
//     MaxIters:  ???,
//     Shots:     1000,
// })
//
// TODO: Print and compare both results:
//   - Amplitude and error vs expected (0.5)
//   - Probability
//   - For iterative: confidence interval and iteration count
//   - Note the resource tradeoff: standard AE uses ancilla qubits, iterative does not
ctx := context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

## Key Takeaways

1. **Shor's algorithm** factors integers in polynomial time by reducing
   factoring to period finding. The quantum speedup comes from QPE on the
   modular exponentiation unitary, followed by continued fraction expansion
   to extract the period.

2. **The order-finding circuit** uses controlled modular exponentiation
   $\text{C-}U^{2^k}$ followed by the inverse QFT. The phase register
   encodes $s/r$ where $r$ is the period, and the factors are recovered via
   $\gcd(a^{r/2} \pm 1, N)$.

3. **Quantum counting** uses QPE on the Grover iterate to estimate the
   number of solutions $M$ in a search space of size $N$. The Grover
   eigenvalue $e^{\pm 2i\theta}$ encodes $M/N = \sin^2(\theta)$.

4. **Amplitude estimation** generalizes quantum counting by estimating the
   probability amplitude of a "good" subspace prepared by an arbitrary
   state-preparation circuit. It is a core primitive for quantum speedups
   in Monte Carlo methods, optimization, and machine learning.

5. **Iterative amplitude estimation** achieves similar precision to standard
   AE without the large ancilla register, trading circuit width for depth
   through progressive Grover power doubling.

---

**Next up:** Notebook 11, where we will explore quantum error correction
and how to protect quantum information from noise.